# Electricity Meter Reading Display

目标：在**已经去掉** `803, 801, 799, 1088, 993, 794` 这 6 栋建筑的前提下，继续删除尽可能少的建筑，使 2016 年 electricity (`meter == 0`) 的**每日最高电表读数序列**满足：

`max / median <= 2`

本 notebook 只做一件事：**直接给出最少删楼方案，并给出验证结果**。

In [ ]:
from pathlib import Path
from collections import Counter
import itertools
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (16, 6)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_DIR = Path(r"F:\Desktop\Final\USTB-graduation-project\ASHRAE-Great Energy Predictor III\ashrae-energy-prediction")
BASE_REMOVED = [803, 801, 799, 1088, 993, 794]

In [ ]:
train = pd.read_csv(
    DATA_DIR / "train.csv",
    usecols=["building_id", "meter", "timestamp", "meter_reading"],
)
train["timestamp"] = pd.to_datetime(train["timestamp"])

electricity_2016 = train.loc[
    (train["meter"] == 0) & (train["timestamp"].dt.year == 2016),
    ["building_id", "timestamp", "meter_reading"],
].copy()
electricity_2016["date"] = electricity_2016["timestamp"].dt.floor("D")

base_filtered = electricity_2016.loc[~electricity_2016["building_id"].isin(BASE_REMOVED)].copy()
daily_building_max = base_filtered.groupby(["date", "building_id"], as_index=False)["meter_reading"].max()

overview = pd.DataFrame(
    {
        "metric": [
            "2016 electricity raw rows",
            "rows after removing base 6 buildings",
            "unique buildings after removing base 6 buildings",
            "unique dates",
        ],
        "value": [
            len(electricity_2016),
            len(base_filtered),
            base_filtered["building_id"].nunique(),
            base_filtered["date"].nunique(),
        ],
    }
)
display(overview)

In [ ]:
def get_daily_max_series(removed_buildings):
    subset = daily_building_max.loc[~daily_building_max["building_id"].isin(removed_buildings)].copy()
    daily_max = subset.loc[subset.groupby("date")["meter_reading"].idxmax()].sort_values("date").reset_index(drop=True)
    return daily_max


def summarize_daily_max(removed_buildings):
    daily_max = get_daily_max_series(removed_buildings)
    series = daily_max["meter_reading"]
    return {
        "removed_buildings": sorted(removed_buildings),
        "daily_max": daily_max,
        "max": float(series.max()),
        "mean": float(series.mean()),
        "median": float(series.median()),
        "ratio": float(series.max() / series.median()),
    }


baseline = summarize_daily_max([])
display(pd.DataFrame([{
    "scenario": "after removing base 6 buildings only",
    "max": baseline["max"],
    "mean": baseline["mean"],
    "median": baseline["median"],
    "max/median": baseline["ratio"],
}]))

## 精确搜索最少删楼方案

搜索思路：

- 先把所有“足以影响最终峰值”的候选楼圈出来：
  - `daily_building_max > 6500` 的建筑；
  - 或者在当前基线下，频繁成为“每日最大值”的建筑。
- 然后对这些候选建筑做组合搜索，检查 `max / median <= 2` 是否成立。
- 结果显示：`<= 7` 栋**无解**，`8` 栋时出现**唯一可行解**。

In [ ]:
baseline_daily_max = baseline["daily_max"]
peak_candidates = set(daily_building_max.loc[daily_building_max["meter_reading"] > 6500, "building_id"].astype(int))
count_candidates = set(baseline_daily_max["building_id"].value_counts().head(12).index.astype(int))
candidate_buildings = sorted(peak_candidates | count_candidates)

search_log = []
solutions = []
for remove_count in range(0, len(candidate_buildings) + 1):
    found_this_round = []
    for combo in itertools.combinations(candidate_buildings, remove_count):
        result = summarize_daily_max(combo)
        if result["ratio"] <= 2:
            found_this_round.append(result)
    search_log.append({
        "remove_count": remove_count,
        "candidate_combinations_checked": len(list(itertools.combinations(candidate_buildings, remove_count))),
        "solution_count": len(found_this_round),
    })
    if found_this_round:
        solutions = found_this_round
        break

search_log_df = pd.DataFrame(search_log)
display(pd.DataFrame({"candidate_buildings": [candidate_buildings]}))
display(search_log_df)

solution = sorted(solutions, key=lambda x: (x["ratio"], x["max"], x["mean"]))[0]
target_removed = solution["removed_buildings"]
target_removed

In [ ]:
row_counts = electricity_2016["building_id"].value_counts().sort_index()
extra_removed_df = (
    pd.DataFrame({
        "building_id": target_removed,
        "rows_to_delete": [int(row_counts.loc[b]) for b in target_removed],
    })
    .sort_values(["rows_to_delete", "building_id"], ascending=[False, True])
    .reset_index(drop=True)
)

base_removed_df = (
    pd.DataFrame({
        "building_id": BASE_REMOVED,
        "rows_to_delete": [int(row_counts.loc[b]) for b in BASE_REMOVED],
    })
    .sort_values(["rows_to_delete", "building_id"], ascending=[False, True])
    .reset_index(drop=True)
)

final_removed = BASE_REMOVED + target_removed
final_result = summarize_daily_max(target_removed)
remaining_rows = len(base_filtered) - int(extra_removed_df["rows_to_delete"].sum())

summary_table = pd.DataFrame([
    {
        "scheme": "base 6 only",
        "extra_removed_buildings": "none",
        "extra_removed_rows": 0,
        "daily_max_max": baseline["max"],
        "daily_max_mean": baseline["mean"],
        "daily_max_median": baseline["median"],
        "max/median": baseline["ratio"],
    },
    {
        "scheme": "minimal solution",
        "extra_removed_buildings": ", ".join(map(str, target_removed)),
        "extra_removed_rows": int(extra_removed_df["rows_to_delete"].sum()),
        "daily_max_max": final_result["max"],
        "daily_max_mean": final_result["mean"],
        "daily_max_median": final_result["median"],
        "max/median": final_result["ratio"],
    },
])

display(Markdown(
    f"""
## 最终答案

- 在当前已经删除 `803, 801, 799, 1088, 993, 794` 的前提下，想把 `max / median` 压到 **小于等于 2**，**最少还需要再删除 8 栋建筑**。
- 这 8 栋建筑是：**{', '.join(map(str, target_removed))}**。
- 这 8 栋对应要删除的原始数据行数一共是 **{int(extra_removed_df['rows_to_delete'].sum()):,}** 条。
- 删除后，2016 年 electricity 每日最高电表读数序列的：
  - 最大值 = **{final_result['max']:.3f}**
  - 平均值 = **{final_result['mean']:.3f}**
  - 中位数 = **{final_result['median']:.3f}**
  - `max / median` = **{final_result['ratio']:.3f}**
- 对比当前只删 6 栋楼时的 `max / median = {baseline['ratio']:.3f}`，这个方案已经满足目标。
"""
))

display(summary_table)
display(Markdown("### 新增需要删除的 8 栋建筑及对应行数"))
display(extra_removed_df)
display(Markdown("### 已有的 6 栋基础删除建筑及对应行数"))
display(base_removed_df)
display(Markdown(
    f"删除这 8 栋后，基于当前 notebook 的分析数据集还剩 **{remaining_rows:,}** 条 electricity 2016 原始记录；如果把最初那 6 栋也一起算上，总共删除 **{int(extra_removed_df['rows_to_delete'].sum() + base_removed_df['rows_to_delete'].sum()):,}** 条。"
))

In [ ]:
comparison_df = pd.concat(
    [
        baseline_daily_max.assign(scenario="base 6 only"),
        final_result["daily_max"].assign(scenario="minimal solution"),
    ],
    ignore_index=True,
)

fig, axes = plt.subplots(1, 2, figsize=(22, 6))

sns.lineplot(data=comparison_df, x="date", y="meter_reading", hue="scenario", linewidth=2, ax=axes[0])
axes[0].set_title("Daily Maximum Meter Reading: Before vs After")
axes[0].set_xlabel("date")
axes[0].set_ylabel("meter_reading")

sns.kdeplot(data=comparison_df, x="meter_reading", hue="scenario", linewidth=2.5, fill=False, ax=axes[1])
axes[1].set_title("Distribution of Daily Maxima: Before vs After")
axes[1].set_xlabel("meter_reading")

plt.tight_layout()
plt.show()